# Outbox heartbeat notebook guide

This notebook is a local smoke test for the template's outbox and worker flow. It starts the development stack, checks that the API is reachable, enqueues a heartbeat job through `POST /jobs/heartbeat`, then listens to the job websocket until the worker marks the job as complete.

## Before you run it

- Run the notebook from this repository so the path setup can import `src` modules.
- Make sure project dependencies are installed with `task sync` if the imports or `task dev` command fail.
- The notebook uses API port `8060` and HTMX port `8061`; the startup cell terminates existing processes on those ports before launching `task dev`.
- Keep the final cleanup cell available so the process group started by `task dev` can be stopped when you are done.

## Run order

Run the cells from top to bottom. First the notebook adds the project root to `sys.path`, then it starts the dev stack, loads settings, checks `/health`, creates a heartbeat job, and finally streams job updates from `/jobs/ws/{job_id}`.

## Expected result

The health check should return `200`. The heartbeat request should return `202` with a `job_id`. The websocket stream should show the job move through `pending`, `running`, and a terminal state such as `done`.

## Troubleshooting

- If the health check fails, wait for the `task dev` startup output to show that the API is running, then rerun the health cell.
- If the heartbeat request fails, inspect the response body printed by the cell and confirm the API server is the one launched by this workspace.
- If the websocket does not close, run the cleanup cell before restarting the notebook flow.


## 1. Prepare notebook imports

This cell adds the project root to `sys.path` so the notebook can import modules from `src` while running from the sandbox directory.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parents[1]
sys.path.append(str(root))

print(f"Root path: {root}")

## 2. Start the local development stack

This cell frees the API and HTMX ports, then launches `task dev` so the API, HTMX app, and worker are available for the smoke test.


In [ ]:
import subprocess
import os
import signal

for port in [8060, 8061]:
    # Get the PID(s) on the port
    result = subprocess.run(["lsof", "-t", f"-i:{port}"], stdout=subprocess.PIPE, text=True)
    pids = result.stdout.strip().split()
    
    for pid in pids:
        if pid:
            try:
                print(f"Stopping process {pid} on port {port}...")
                os.kill(int(pid), signal.SIGTERM) # Clean kill
            except ProcessLookupError:
                pass

dir = os.path.expanduser("~/Projects/homebrew/template")
process = subprocess.Popen(
    ["task", "dev"],
    cwd=dir,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

## 3. Load runtime settings

This cell reads the application settings and builds the API base URL used by the HTTP checks below.


In [ ]:
from infrastructure.config import get_settings

settings = get_settings()
print(f"Settings loaded: {settings.model_dump_json(indent=2)}")

base_url = f"http://{settings.api_host}:{settings.api_port}"

## 4. Check API health

This cell calls `/health` to confirm the API process is accepting requests before creating a job.


In [ ]:
import httpx

with httpx.Client(base_url=base_url) as client:
    response = client.get("/health")
    if response.status_code == 200:
        print("✅ API is up and running!")
    else:
        print(f"❌ API health check failed with status code: {response.status_code}")

## 5. Enqueue a heartbeat job

This cell posts to `/jobs/heartbeat` and expects a `202 Accepted` response with a `job_id`.


In [ ]:
with httpx.Client(base_url=base_url) as client:
    # 1. Pass an empty dict (or actual heartbeat values) to satisfy Pydantic
    response = client.post("/jobs/heartbeat", json={
        "beats": 5,
        "interval": 2.5
    })
    
    # 2. Check for 202 Accepted instead of 200 OK
    if response.status_code == 202:
        print("✅ Job heartbeat is active!")
        print(f"Heartbeat response: {response.json()}")
    else:
        print(f"❌ Job heartbeat check failed with status code: {response.status_code}")
        print(f"Response details: {response.text}")

## 6. Stream job updates

This cell creates a heartbeat job, opens `/jobs/ws/{job_id}`, and prints status updates until the worker reaches a terminal state.


In [ ]:
import json
import websockets

def heartbeat(beats = 5 , interval = 10):
    with httpx.Client(base_url=base_url) as client:
        response = client.post("/jobs/heartbeat", json={
            "beats": beats,
            "interval": interval
        })
        
        if response.status_code == 202:
            response = response.json()
            assert 'job_id' in response
            return response['job_id']
        else:
            print(f"❌ Job heartbeat check failed with status code: {response.status_code}")
            print(f"Response details: {response.text}")


job_id = heartbeat()

URI = f"ws://{settings.api_host}:{settings.api_port}/jobs/ws/{job_id}"
try:
    async with websockets.connect(URI) as websocket:
        async for message in websocket:
            data = json.loads(message)
  
            print(f"[{data.get('status').upper()}] - Updated at: {data.get('finished_at') or data.get('started_at') or 'Just now'}")
            print(json.dumps(data, indent=2))
            print("-" * 40)

except websockets.exceptions.ConnectionClosedOK:
    print("\n🏁 Connection closed cleanly: Job reached a terminal state.")
except Exception as e:
    print(f"\n❌ Error: {e}")

## 7. Stop the development stack

Run this cleanup cell when you are finished so the `task dev` process group does not keep running in the background.


In [ ]:
import os
import signal

pgid = os.getpgid(process.pid)
os.killpg(pgid, signal.SIGKILL)